# Classical ML Models

## Objective

This notebook evaluates classical machine learning approaches for privacy sensitive prompt detection.

Labels:
- 0 = safe (no PII)
- 1 = privacy sensitive (contains PII)

This notebook trains and evaluates the following models:
- Logistic Regression
- Naive Bayes
- Linear SVM

## Experiment Design

To keep the model comparison controlled, the models use the same frozen train, validation, and test splits and the same combined feature representation produced by `02_feature_engineering.ipynb`. The unified representation makes differences between models easier to interpret, although it may not be the individually most optimal feature representation for each model.

The evaluation procedure is:
1. Fit models on training split
2. Select model hyperparameters using the validation split
3. Select a separate decision threshold for each fitted model
4. Evaluate each final config once on the held out test split

The primary metric is F1 for the PII class. Additionally precision, recall, accuracy and confusion matrices are also produced. False negatives deserve additional attention here since they represent privacy sensitive prompts that would be allowed through inadvertantly marked as safe.

A fixed random seed of 42 is used as needed to support reproducibility.

In [1]:
# Imports
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, ComplementNB
from sklearn.svm import LinearSVC 
from pathlib import Path
import numpy as np
from scipy.sparse import load_npz
from sklearn.base import clone
from sklearn.model_selection import ParameterGrid
import time
import os
import json
import sys, subprocess
from scipy.sparse import hstack, csr_matrix
import joblib

## Load Frozen Data Splits
Train, validation, and test sets generated by `02_feature_engineering.ipynb`.

In [2]:
# Colab compatible setup referenced from distilbert notebook
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo_url = 'https://github.com/Cyber-207/cyber207_specialized_PII_detection_comparison.git'
    repo_name = 'cyber207_specialized_PII_detection_comparison'
    %cd /content
    if not Path(repo_name).exists():
        !git clone {repo_url}
    else:
        print('Repo already cloned.')
    %cd /content/{repo_name}

repo_root = subprocess.run(
    ['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True
).stdout.strip()
repo_root = Path(repo_root)
print('Repo root:', repo_root)

# make repo modules importable
sys.path.insert(0, str(repo_root))

from src.features.text_features import extract_text_features
from src.features.pattern_features import extract_pattern_features

FEATURE_DIR = repo_root / 'feature_matrices'

train_features = load_npz(FEATURE_DIR / 'X_train_combined.npz')
val_features = load_npz(FEATURE_DIR / 'X_val_combined.npz')
test_features = load_npz(FEATURE_DIR / 'X_test_combined.npz')

train_labels = np.load(FEATURE_DIR / 'y_train.npy')
val_labels = np.load(FEATURE_DIR / 'y_val.npy')
test_labels = np.load(FEATURE_DIR / 'y_test.npy')

Repo root: ~/cyber207_specialized_PII_detection_comparison


In [3]:
# data set verification sanity check
print(train_features.shape)
print(val_features.shape)
print(test_features.shape)

(260338, 50018)
(32542, 50018)
(32543, 50018)


## Majority Class Baseline: Always Predict PII

The PII class is mroe common than the safe class in the data set, so a classifier can obtain deceptively high metrics by labeling every prompt as PII.

As a simple baseline, we evaluate a model that predicts every prompt as PII (class 1). This baseline helps show whether the trained models are learning beyond the class imbalance.

In [4]:
# create array of 1s to serve as the baseline
baseline_predictions = np.ones(len(val_labels), dtype=int)
# zero_division set to 0 to hide warning due to zero examples being predicted as safe
print(classification_report(val_labels, baseline_predictions, zero_division=0))

cm = confusion_matrix(val_labels, baseline_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     10616
           1       0.67      1.00      0.81     21926

    accuracy                           0.67     32542
   macro avg       0.34      0.50      0.40     32542
weighted avg       0.45      0.67      0.54     32542



,Predicted Safe,Predicted PII
Actual Safe,0,10616
Actual PII,0,21926


## Logistic Regression

Logistic Regression is a common baseline for text classification because it performs well on sparse TF-IDF representations and produces interpretable feature weights. However, the model learns a linear decision boundary and may struggle to capture complex relationships between terms.





The model computes a weighted sum of the input features:

$$
z = \sum_{i=1}^{n} w_i x_i + b
$$

The sigmoid function maps this score to the range [0, 1]:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

The resulting value is interpreted as the probability that a prompt belongs to the privacy sensitive class:

$$
P(y=1|x) = \sigma(z)
$$

Reference: 

[scikit-learn - Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

[geekforgeeks - Understanding Logistic Regression](https://www.geeksforgeeks.org/machine-learning/understanding-logistic-regression/)


### Hyperparameter Tuning

The tuning helper function performs a manual held out validation search where:
- Each config is fit using only the training split
- Configs are ranked by PII class F1
- Results are written to a CSV file so interrupted searches can resume without repeating completed runs

For Logistic Regression, the search compares different values of C and L1 / L2 regularization. C is the inverse regularization strength (smaller values apply stronger regularization).

The final selected config is `C=2.65` with `L1 regularization`. Several nearby values also produce similar validation results, indicating that performance is relatively stable around the selected value.

In [5]:
# leaving outside for if combined comparison is made later
results = []
save_path = '../results/classical_tuning_results.csv'

# load data from csv
if os.path.exists(save_path):
    results_df = pd.read_csv(save_path)
    results = results_df.to_dict('records')
    completed_keys = set(results_df['key'])
    print(f'Loaded {len(results)} saved results from CSV.')
else:
    completed_keys = set()
    print('No saved results found.')

# helper function to make keys
def make_key(model_name, params):
    return f'{model_name}|{json.dumps(params, sort_keys=True)}'

# helper function for tuning the classical models

def run_search(model_name, base_model, param_grid):
    configs = list(ParameterGrid(param_grid))

    print(f'\n----- {model_name} tuning started -----')
    print(f'Total configs for {model_name}: {len(configs)}')

    for i, params in enumerate(configs, start=1):
        # if already in csv, skip to save time
        key = make_key(model_name, params)

        if key in completed_keys:
            print(f'Skipping {model_name} {i}/{len(configs)}: already saved')
            continue

        print('\n' + '-' * 60)
        print(f'{model_name} progress: {i}/{len(configs)}')
        print(f'Params: {params}')

        # clone the base model and use this iteration's params
        model = clone(base_model)
        model.set_params(**params)
        
        # keep track of time to find bottlenecks
        start = time.perf_counter()

        # train model
        model.fit(train_features, train_labels)
        preds = model.predict(val_features)

        elapsed = (time.perf_counter() - start) / 60

        row = {
            'model': model_name,
            'params': str(params),
            'key': key,
            **params,
            'accuracy': accuracy_score(val_labels, preds),
            'precision': precision_score(val_labels, preds),
            'recall': recall_score(val_labels, preds),
            'f1': f1_score(val_labels, preds),
            'time_min': elapsed
        }

        results.append(row)
        completed_keys.add(key)
        # save to csv so that this doesn't need to be run everytime
        results_df = pd.DataFrame(results)
        results_df = results_df.drop_duplicates(subset=['key'], keep='last')
        results_df.to_csv(save_path, index=False)

        results_df = pd.DataFrame(results).sort_values('f1', ascending=False)
        # best = results_df.iloc[0]

        # progress tracking (made the mistake of not doing this last time)
        print(
            f"Finished in {elapsed:.2f} min\n"
            f"F1:        {row['f1']:.4f}\n"
            f"Precision: {row['precision']:.4f}\n"
            f"Recall:    {row['recall']:.4f}"
        )

        # print(f'Best overall so far: {best['model']} | F1={best['f1']:.4f}')

    print(f'\n----- {model_name} tuning complete -----')

    model_results = (pd.DataFrame(results).query('model == @model_name').sort_values('f1', ascending=False))

    print(f'\nTop results for {model_name}:')
    display(model_results.head(10))

Loaded 114 saved results from CSV.


In [6]:
# saga with elastic was taking too long so the solver has been switched over to liblinear
run_search(
    model_name='Logistic Regression',
    base_model=LogisticRegression(solver='liblinear', max_iter=1000, random_state=42),
    param_grid={'C': [0.1, 1, 2.5, 2.6, 2.65, 2.7, 2.75, 2.8, 2.85, 2.9, 2.95, 3, 3.25, 3.5, 5, 10], 'l1_ratio': [0.0, 1.0]}
)


----- Logistic Regression tuning started -----
Total configs for Logistic Regression: 32
Skipping Logistic Regression 1/32: already saved
Skipping Logistic Regression 2/32: already saved
Skipping Logistic Regression 3/32: already saved
Skipping Logistic Regression 4/32: already saved
Skipping Logistic Regression 5/32: already saved
Skipping Logistic Regression 6/32: already saved
Skipping Logistic Regression 7/32: already saved
Skipping Logistic Regression 8/32: already saved
Skipping Logistic Regression 9/32: already saved
Skipping Logistic Regression 10/32: already saved
Skipping Logistic Regression 11/32: already saved
Skipping Logistic Regression 12/32: already saved
Skipping Logistic Regression 13/32: already saved
Skipping Logistic Regression 14/32: already saved
Skipping Logistic Regression 15/32: already saved
Skipping Logistic Regression 16/32: already saved
Skipping Logistic Regression 17/32: already saved
Skipping Logistic Regression 18/32: already saved
Skipping Logistic R

,model,params,key,C,l1_ratio,accuracy,precision,recall,f1,time_min,alpha,fit_prior,norm,loss
11,Logistic Regression,"{'C': 2.7, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.7, ""l1_ratio"": 1.0}",2.70,1.0,0.829912,0.869017,0.880234,0.874589,0.765765,NaN,NaN,NaN,NaN
7,Logistic Regression,"{'C': 2.6, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.6, ""l1_ratio"": 1.0}",2.60,1.0,0.829820,0.868634,0.880598,0.874575,0.750823,NaN,NaN,NaN,NaN
9,Logistic Regression,"{'C': 2.65, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.65, ""l1_ratio"": 1.0}",2.65,1.0,0.829697,0.868644,0.880370,0.874468,0.749927,NaN,NaN,NaN,NaN
13,Logistic Regression,"{'C': 2.75, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.75, ""l1_ratio"": 1.0}",2.75,1.0,0.829697,0.868876,0.880051,0.874428,0.776741,NaN,NaN,NaN,NaN
5,Logistic Regression,"{'C': 2.5, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.5, ""l1_ratio"": 1.0}",2.50,1.0,0.829605,0.868659,0.880188,0.874386,0.752043,NaN,NaN,NaN,NaN
15,Logistic Regression,"{'C': 2.8, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.8, ""l1_ratio"": 1.0}",2.80,1.0,0.829574,0.868786,0.879960,0.874337,0.767703,NaN,NaN,NaN,NaN
23,Logistic Regression,"{'C': 3, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 3, ""l1_ratio"": 1.0}",3.00,1.0,0.829144,0.868305,0.879869,0.874049,0.730985,NaN,NaN,NaN,NaN
17,Logistic Regression,"{'C': 2.85, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.85, ""l1_ratio"": 1.0}",2.85,1.0,0.829082,0.868359,0.879686,0.873986,0.759764,NaN,NaN,NaN,NaN
21,Logistic Regression,"{'C': 2.95, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.95, ""l1_ratio"": 1.0}",2.95,1.0,0.829082,0.868359,0.879686,0.873986,0.822897,NaN,NaN,NaN,NaN
19,Logistic Regression,"{'C': 2.9, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.9, ""l1_ratio"": 1.0}",2.90,1.0,0.828959,0.868269,0.879595,0.873896,0.778090,NaN,NaN,NaN,NaN


### Training

Fit the logistic regression model using the engineered combined feature representation of the training data.
Use set seed (42) to ensure reproducible results across runs / team members. 

In [7]:
# reused seed used in data split script
lr = LogisticRegression(C=2.65, l1_ratio=1, random_state=42, max_iter=1000, solver='liblinear')
lr.fit(train_features, train_labels)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",2.65
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;-

### Threshold Selection

Logistic Regression by default predicts PII when the estimated class 1 probability is at least 0.5.

Here, a continuous PII probability for every validation example is produced. Then using the function precision_recall_curve to evaluate, the threshold with the highest validation F1 is selected.

Lowering the threshold generally predicts more prompts as PII, which tends to increase recall and reduce false negatives at the cost of additional false positives. Raising it has the opposite effect.

In [8]:
def find_best_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)

    # remove final value with no threshold
    precision = precision[:-1]
    recall = recall[:-1]

    # calculate f1 for every threshold
    # zeros when calculation can't be performed and only divide when precision + recall is not 0
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(precision), where=(precision + recall) != 0)

    # index of highest f1
    best_index = np.argmax(f1)
    best_threshold = thresholds[best_index]

    results = pd.DataFrame({'threshold': thresholds, 'precision': precision, 'recall': recall, 'f1': f1}).sort_values('f1', ascending=False)

    return best_threshold, results

In [9]:
lr_scores = lr.predict_proba(val_features)[:, 1]
lr_threshold = find_best_threshold(val_labels, lr_scores)[0]
print('Best threshold: ', lr_threshold)
display(find_best_threshold(val_labels, lr_scores)[1].head(10))

Best threshold:  0.41589660496670616


,threshold,precision,recall,f1
8910,0.415897,0.845119,0.910836,0.876748
8880,0.413975,0.844597,0.911429,0.876741
8921,0.416464,0.845301,0.910608,0.876740
8909,0.415791,0.845083,0.910836,0.876729
8918,0.416425,0.845236,0.910654,0.876726
8927,0.416787,0.845388,0.910472,0.876724
8911,0.415908,0.845112,0.910791,0.876723
8895,0.414686,0.844836,0.911110,0.876723
8879,0.413938,0.844561,0.911429,0.876722
8920,0.416456,0.845265,0.910608,0.876721


### Validation Evaluation

Generate predictions on the validation set for model evaluation, then evaluate model performance using a classification report and confusion matrix.

The most security relevant error is the false negative. A false negative represents a privacy sensitive prompt that the detector fails to block.

[![Confusion Matrix](https://i0.wp.com/statisticsbyjim.com/wp-content/uploads/2025/05/confusion_matrix-1.png?fit=550%2C450&ssl=1)](https://statisticsbyjim.com/glossary/confusion-matrix/)

In [10]:
lr_predictions = (lr_scores >= lr_threshold).astype(int)
print(classification_report(val_labels, lr_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(val_labels, lr_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.78      0.66      0.71     10616
         PII       0.85      0.91      0.88     21926

    accuracy                           0.83     32542
   macro avg       0.81      0.78      0.79     32542
weighted avg       0.82      0.83      0.82     32542



,Predicted Safe,Predicted PII
Actual Safe,6956,3660
Actual PII,1955,19971


## Logistic Regression Feature Ablation

Train the same Logistic Regression configuration on each feature matrix saved by the feature engineering notebook to compare the contribution of each feature set.

The validation experiments are:
- TF-IDF only
- Text only
- Pattern only
- TF-IDF + text shape features
- TF-IDF + privacy pattern features
- All features: TF-IDF + text-shape + privacy-pattern features

The model configuration and default threshold of 0.5 remain the same for every run. Only the feature representation changes, and all results are evaluated on the validation set.

In [11]:
# load saved matrices
tfidf_train = load_npz(FEATURE_DIR / 'X_train_tfidf.npz')
tfidf_val = load_npz(FEATURE_DIR / 'X_val_tfidf.npz')

text_train = load_npz(FEATURE_DIR / 'X_train_text.npz')
text_val = load_npz(FEATURE_DIR / 'X_val_text.npz')

pattern_train = load_npz(FEATURE_DIR / 'X_train_pattern.npz')
pattern_val = load_npz(FEATURE_DIR / 'X_val_pattern.npz')

combined_train = load_npz(FEATURE_DIR / 'X_train_combined.npz')
combined_val = load_npz(FEATURE_DIR / 'X_val_combined.npz')

feature_sets = {
    'tfidf': (tfidf_train, tfidf_val),
    'text': (text_train, text_val),
    'pattern': (pattern_train, pattern_val),
    'tfidf + text': (hstack([tfidf_train, text_train]).tocsr(), hstack([tfidf_val, text_val]).tocsr()),
    'tfidf + pattern': (hstack([tfidf_train, pattern_train]).tocsr(), hstack([tfidf_val, pattern_val]).tocsr()),
    'combined': (combined_train, combined_val)
}

ablation_results = []

for feature_set, (train, val) in feature_sets.items():
    # train the same logistic regression config
    model = clone(lr)
    model.fit(train, train_labels)
    predictions = model.predict(val)

    tn, fp, fn, tp = confusion_matrix(val_labels, predictions).ravel()

    ablation_results.append({
        'Feature Set': feature_set,
        'Feature Count': train.shape[1],
        'Accuracy': accuracy_score(val_labels, predictions),
        'Precision': precision_score(val_labels, predictions),
        'Recall': recall_score(val_labels, predictions),
        'F1': f1_score(val_labels, predictions),
        'False Positives': fp,
        'False Negatives': fn
    })

pd.DataFrame(ablation_results)

,Feature Set,Feature Count,Accuracy,Precision,Recall,F1,False Positives,False Negatives
0,tfidf,50000,0.813841,0.855370,0.870975,0.863102,3229,2829
1,text,10,0.696730,0.709753,0.930357,0.805218,8342,1527
2,pattern,8,0.673775,0.673775,1.000000,0.805097,10616,0
3,tfidf + text,50010,0.828744,0.867599,0.880142,0.873826,2945,2628
4,tfidf + pattern,50008,0.825549,0.865557,0.877360,0.871419,2988,2689
5,combined,50018,0.829697,0.868644,0.880370,0.874468,2919,2623


### Ablation Results

The engineered features provided complementary information beyond just the TF-IDF representation. The TF-IDF-only model achieved a F1 score of 0.8632. Adding the basic text features increased F1 to 0.8743, while adding the privacy-pattern features increased F1 to 0.8714.

The full combined representation achieved the highest overall validation performance, with an accuracy of 0.8297, precision of 0.8686, recall of 0.8804, and F1 score of 0.8745. Compared with TF-IDF alone, the combined features increased F1 by 0.0112 and reduced false positives from 3,223 to 2,919 and false negatives from 2,829 to 2,623.

These results suggest that the text shape and privacy pattern features capture signals complementary to the information represented by TF-IDF.

## Naive Bayes

Multinomial Naive Bayes is a probabilistic classification algorithm also commonly used for text classification.

The model estimates the probability that a prompt belongs to each class based on the observed features and predicts the class with the highest posterior probability.

Bayes' Theorem:

$$
P(y|x)=\frac{P(x|y)P(y)}{P(x)}
$$

Naive Bayes assumes that features are conditionally independent given the class label.

$$
P(x_1,x_2,\ldots,x_n|y)
=
\prod_{i=1}^{n} P(x_i|y)
$$

In other words, once the model knows whether a prompt belongs to the Safe or PII class, it treats each feature as contributing independently to the final prediction. While this assumption is rarely true for real-world text data, it greatly simplifies computation and often performs surprisingly well for text classification tasks.

Reference:

[scikit-learn - Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)

[scikit-learn - Naive Bayes Documentation](https://scikit-learn.org/stable/api/sklearn.naive_bayes.html)

### Hyperparameter Tuning

The Naive Bayes search compares:
- MultinomialNB: smoothing parameter alpha
- BernoulliNB: smoothing parameter alpha
- ComplementNB: alpha, class prior fitting, and normalization

The smoothing parameter (alpha) prevents unseen / rare features from recieving zero probability.

Multinomial Naive Bayes with `alpha=0.0001` was selected as the final Naive Bayes config.

In [12]:
run_search(
    model_name='MultinomialNB',
    base_model=MultinomialNB(),
    param_grid={'alpha': [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0]}
)

run_search(
    model_name='BernoulliNB',
    base_model=BernoulliNB(binarize=0.0),
    param_grid={'alpha': [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0]}
)

run_search(
    model_name='ComplementNB',
    base_model=ComplementNB(),
    param_grid={'alpha': [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1], 'fit_prior': [True, False], 'norm': [False, True]}
)


----- MultinomialNB tuning started -----
Total configs for MultinomialNB: 7
Skipping MultinomialNB 1/7: already saved
Skipping MultinomialNB 2/7: already saved
Skipping MultinomialNB 3/7: already saved
Skipping MultinomialNB 4/7: already saved
Skipping MultinomialNB 5/7: already saved
Skipping MultinomialNB 6/7: already saved
Skipping MultinomialNB 7/7: already saved

----- MultinomialNB tuning complete -----

Top results for MultinomialNB:


,model,params,key,C,l1_ratio,accuracy,precision,recall,f1,time_min,alpha,fit_prior,norm,loss
32,MultinomialNB,{'alpha': 0.0001},"MultinomialNB|{""alpha"": 0.0001}",NaN,NaN,0.750507,0.808261,0.825550,0.816814,0.000877,0.0001,NaN,NaN,NaN
33,MultinomialNB,{'alpha': 0.001},"MultinomialNB|{""alpha"": 0.001}",NaN,NaN,0.750323,0.808237,0.825230,0.816645,0.000836,0.0010,NaN,NaN,NaN
34,MultinomialNB,{'alpha': 0.01},"MultinomialNB|{""alpha"": 0.01}",NaN,NaN,0.749862,0.808136,0.824501,0.816236,0.000838,0.0100,NaN,NaN,NaN
35,MultinomialNB,{'alpha': 0.05},"MultinomialNB|{""alpha"": 0.05}",NaN,NaN,0.749124,0.807957,0.823360,0.815586,0.000822,0.0500,NaN,NaN,NaN
36,MultinomialNB,{'alpha': 0.1},"MultinomialNB|{""alpha"": 0.1}",NaN,NaN,0.748755,0.807661,0.823132,0.815323,0.000866,0.1000,NaN,NaN,NaN
38,MultinomialNB,{'alpha': 1.0},"MultinomialNB|{""alpha"": 1.0}",NaN,NaN,0.744884,0.798589,0.830931,0.814439,0.000823,1.0000,NaN,NaN,NaN
37,MultinomialNB,{'alpha': 0.5},"MultinomialNB|{""alpha"": 0.5}",NaN,NaN,0.746082,0.804014,0.823999,0.813884,0.000828,0.5000,NaN,NaN,NaN



----- BernoulliNB tuning started -----
Total configs for BernoulliNB: 7
Skipping BernoulliNB 1/7: already saved
Skipping BernoulliNB 2/7: already saved
Skipping BernoulliNB 3/7: already saved
Skipping BernoulliNB 4/7: already saved
Skipping BernoulliNB 5/7: already saved
Skipping BernoulliNB 6/7: already saved
Skipping BernoulliNB 7/7: already saved

----- BernoulliNB tuning complete -----

Top results for BernoulliNB:


,model,params,key,C,l1_ratio,accuracy,precision,recall,f1,time_min,alpha,fit_prior,norm,loss
39,BernoulliNB,{'alpha': 0.0001},"BernoulliNB|{""alpha"": 0.0001}",NaN,NaN,0.735695,0.856455,0.730092,0.788241,0.001528,0.0001,NaN,NaN,NaN
40,BernoulliNB,{'alpha': 0.001},"BernoulliNB|{""alpha"": 0.001}",NaN,NaN,0.735542,0.856493,0.729773,0.788071,0.001605,0.0010,NaN,NaN,NaN
41,BernoulliNB,{'alpha': 0.01},"BernoulliNB|{""alpha"": 0.01}",NaN,NaN,0.734958,0.856538,0.728678,0.787452,0.001494,0.0100,NaN,NaN,NaN
42,BernoulliNB,{'alpha': 0.05},"BernoulliNB|{""alpha"": 0.05}",NaN,NaN,0.733821,0.856521,0.726672,0.786271,0.001746,0.0500,NaN,NaN,NaN
43,BernoulliNB,{'alpha': 0.1},"BernoulliNB|{""alpha"": 0.1}",NaN,NaN,0.732684,0.856312,0.724893,0.785141,0.001629,0.1000,NaN,NaN,NaN
44,BernoulliNB,{'alpha': 0.5},"BernoulliNB|{""alpha"": 0.5}",NaN,NaN,0.730594,0.856323,0.721153,0.782947,0.001524,0.5000,NaN,NaN,NaN
45,BernoulliNB,{'alpha': 1.0},"BernoulliNB|{""alpha"": 1.0}",NaN,NaN,0.729611,0.855687,0.720150,0.782090,0.001480,1.0000,NaN,NaN,NaN



----- ComplementNB tuning started -----
Total configs for ComplementNB: 24
Skipping ComplementNB 1/24: already saved
Skipping ComplementNB 2/24: already saved
Skipping ComplementNB 3/24: already saved
Skipping ComplementNB 4/24: already saved
Skipping ComplementNB 5/24: already saved
Skipping ComplementNB 6/24: already saved
Skipping ComplementNB 7/24: already saved
Skipping ComplementNB 8/24: already saved
Skipping ComplementNB 9/24: already saved
Skipping ComplementNB 10/24: already saved
Skipping ComplementNB 11/24: already saved
Skipping ComplementNB 12/24: already saved
Skipping ComplementNB 13/24: already saved
Skipping ComplementNB 14/24: already saved
Skipping ComplementNB 15/24: already saved
Skipping ComplementNB 16/24: already saved
Skipping ComplementNB 17/24: already saved
Skipping ComplementNB 18/24: already saved
Skipping ComplementNB 19/24: already saved
Skipping ComplementNB 20/24: already saved
Skipping ComplementNB 21/24: already saved
Skipping ComplementNB 22/24: a

,model,params,key,C,l1_ratio,accuracy,precision,recall,f1,time_min,alpha,fit_prior,norm,loss
46,ComplementNB,"{'alpha': 0.0001, 'fit_prior': True, 'norm': F...","ComplementNB|{""alpha"": 0.0001, ""fit_prior"": tr...",NaN,NaN,0.734036,0.878544,0.702362,0.780636,0.000823,0.0001,True,False,NaN
48,ComplementNB,"{'alpha': 0.0001, 'fit_prior': False, 'norm': ...","ComplementNB|{""alpha"": 0.0001, ""fit_prior"": fa...",NaN,NaN,0.734036,0.878544,0.702362,0.780636,0.000831,0.0001,False,False,NaN
52,ComplementNB,"{'alpha': 0.001, 'fit_prior': False, 'norm': F...","ComplementNB|{""alpha"": 0.001, ""fit_prior"": fal...",NaN,NaN,0.733514,0.878642,0.701359,0.780055,0.000759,0.0010,False,False,NaN
50,ComplementNB,"{'alpha': 0.001, 'fit_prior': True, 'norm': Fa...","ComplementNB|{""alpha"": 0.001, ""fit_prior"": tru...",NaN,NaN,0.733514,0.878642,0.701359,0.780055,0.000769,0.0010,True,False,NaN
56,ComplementNB,"{'alpha': 0.005, 'fit_prior': False, 'norm': F...","ComplementNB|{""alpha"": 0.005, ""fit_prior"": fal...",NaN,NaN,0.732807,0.878699,0.700082,0.779287,0.000793,0.0050,False,False,NaN
54,ComplementNB,"{'alpha': 0.005, 'fit_prior': True, 'norm': Fa...","ComplementNB|{""alpha"": 0.005, ""fit_prior"": tru...",NaN,NaN,0.732807,0.878699,0.700082,0.779287,0.000778,0.0050,True,False,NaN
58,ComplementNB,"{'alpha': 0.01, 'fit_prior': True, 'norm': False}","ComplementNB|{""alpha"": 0.01, ""fit_prior"": true...",NaN,NaN,0.732346,0.878812,0.699170,0.778766,0.000765,0.0100,True,False,NaN
60,ComplementNB,"{'alpha': 0.01, 'fit_prior': False, 'norm': Fa...","ComplementNB|{""alpha"": 0.01, ""fit_prior"": fals...",NaN,NaN,0.732346,0.878812,0.699170,0.778766,0.000776,0.0100,False,False,NaN
67,ComplementNB,"{'alpha': 0.1, 'fit_prior': True, 'norm': True}","ComplementNB|{""alpha"": 0.1, ""fit_prior"": true,...",NaN,NaN,0.730410,0.875221,0.699626,0.777634,0.000757,0.1000,True,True,NaN
69,ComplementNB,"{'alpha': 0.1, 'fit_prior': False, 'norm': True}","ComplementNB|{""alpha"": 0.1, ""fit_prior"": false...",NaN,NaN,0.730410,0.875221,0.699626,0.777634,0.000776,0.1000,False,True,NaN


### Training

Fit the Naive Bayes model using the engineered combined feature representation of the training data.

In [13]:
nb = MultinomialNB(alpha=0.0001)
nb.fit(train_features, train_labels)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",0.0001
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[ 84930.,175408.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-1.12,-0.39]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 50018)","[[ 382.97, 151.5 , 1.79,..., 2921. , 529. , 528. ], [ 1306.57, 596.47, 5.13,...,12122. , 6996. , 3669. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 50018)","[[ -7.19, -8.12,-12.56,..., -5.16, -6.87, -6.87], [ -6.87, -7.65,-12.41,..., -4.64, -5.19, -5.83]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,50018


### Threshold Selection

Multinomial Naive Bayes produces an estimated probability for the PII class with a default threshold of 0.5.

As with Logistic Regression, the threshold is selected based on the highest PII F1.

In [14]:
nb_scores = nb.predict_proba(val_features)[:, 1]
nb_threshold = find_best_threshold(val_labels, nb_scores)[0]
print('Best threshold: ', nb_threshold)
display(find_best_threshold(val_labels, nb_scores)[1].head(10))

Best threshold:  0.33209003413769805


,threshold,precision,recall,f1
5358,0.332090,0.746984,0.926115,0.826960
5348,0.331814,0.746856,0.926298,0.826954
5343,0.331581,0.746792,0.926389,0.826952
5355,0.332036,0.746938,0.926161,0.826950
5357,0.332080,0.746956,0.926115,0.826943
5347,0.331749,0.746828,0.926298,0.826938
5359,0.332098,0.746974,0.926070,0.826936
5342,0.331580,0.746765,0.926389,0.826935
5325,0.330986,0.746555,0.926708,0.826934
5354,0.332027,0.746910,0.926161,0.826933


### Evaluation

Generate predictions on the validation set for model evaluation, then evaluate model performance using a classification report and confusion matrix.

Naive Bayes attains high PII recall but produces substantially more false positives than Logistic Regression or Linear SVM.

In [15]:
nb_predictions = (nb_scores >= nb_threshold).astype(int)
print(classification_report(val_labels, nb_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(val_labels, nb_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.70      0.35      0.47     10616
         PII       0.75      0.93      0.83     21926

    accuracy                           0.74     32542
   macro avg       0.72      0.64      0.65     32542
weighted avg       0.73      0.74      0.71     32542



,Predicted Safe,Predicted PII
Actual Safe,3738,6878
Actual PII,1620,20306


## Linear Support Vector Machine (SVM)

A Linear Support Vector Machine is a classification model that tries to find a decision boundary separating the Safe and PII classes with the largest possible margin.

For a linear decision boundary, the model computes:

$$
f(x) = w^T x + b
$$

- Class 0: f(x) < 0
- Class 1: f(x) > 0

Predictions are based on which side of the decision boundary the example falls on.

Unlike Logistic Regression and Naive Bayes, the decision_function output is a signed margin score rather than a probability. Larger positive values indicate stronger evidence for PII, while more negative values indicate stronger evidence for Safe.

Reference:

[scikit-learn - LinearSVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)

[geekforgeeks - Support Vector Machine Algorithm](https://www.geeksforgeeks.org/machine-learning/support-vector-machine-algorithm/)


### Hyperparameter Tuning

The Linear SVM search evaluates:
- different values of `C`
- `hinge` and `square_hinge` loss functions

C controls the regularization tradeoff as mentioned.

The final config uses `C=1.5` with `hinge` loss.

In [16]:
run_search(
    model_name='LinearSVM',
    base_model=LinearSVC(dual='auto', max_iter=5000, random_state=42),
    param_grid={'C': [0.01, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 1, 1.1, 1.2, 1.25, 1.4, 1.5, 1.6, 1.8, 2]}
)

run_search(
    model_name='LinearSVM Loss Check',
    base_model=LinearSVC(dual='auto', max_iter=5000, random_state=42),
    param_grid={'C': [0.01, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 1, 1.1, 1.2, 1.25, 1.5, 1.75, 2], 'loss': ['hinge', 'squared_hinge']}
)


----- LinearSVM tuning started -----
Total configs for LinearSVM: 16
Skipping LinearSVM 1/16: already saved
Skipping LinearSVM 2/16: already saved
Skipping LinearSVM 3/16: already saved
Skipping LinearSVM 4/16: already saved
Skipping LinearSVM 5/16: already saved
Skipping LinearSVM 6/16: already saved
Skipping LinearSVM 7/16: already saved
Skipping LinearSVM 8/16: already saved
Skipping LinearSVM 9/16: already saved
Skipping LinearSVM 10/16: already saved
Skipping LinearSVM 11/16: already saved
Skipping LinearSVM 12/16: already saved
Skipping LinearSVM 13/16: already saved
Skipping LinearSVM 14/16: already saved
Skipping LinearSVM 15/16: already saved
Skipping LinearSVM 16/16: already saved

----- LinearSVM tuning complete -----

Top results for LinearSVM:


,model,params,key,C,l1_ratio,accuracy,precision,recall,f1,time_min,alpha,fit_prior,norm,loss
72,LinearSVM,{'C': 0.25},"LinearSVM|{""C"": 0.25}",0.25,NaN,0.827177,0.863009,0.883791,0.873276,0.225917,NaN,NaN,NaN,NaN
73,LinearSVM,{'C': 0.5},"LinearSVM|{""C"": 0.5}",0.50,NaN,0.827331,0.865677,0.880325,0.872939,0.304930,NaN,NaN,NaN,NaN
74,LinearSVM,{'C': 0.75},"LinearSVM|{""C"": 0.75}",0.75,NaN,0.827331,0.866829,0.878728,0.872738,0.328835,NaN,NaN,NaN,NaN
75,LinearSVM,{'C': 0.8},"LinearSVM|{""C"": 0.8}",0.80,NaN,0.827300,0.867021,0.878409,0.872678,0.339868,NaN,NaN,NaN,NaN
76,LinearSVM,{'C': 0.9},"LinearSVM|{""C"": 0.9}",0.90,NaN,0.827054,0.867237,0.877679,0.872427,0.363603,NaN,NaN,NaN,NaN
77,LinearSVM,{'C': 1},"LinearSVM|{""C"": 1}",1.00,NaN,0.826286,0.866988,0.876676,0.871805,0.486740,NaN,NaN,NaN,NaN
78,LinearSVM,{'C': 1.1},"LinearSVM|{""C"": 1.1}",1.10,NaN,0.825426,0.866423,0.875946,0.871159,0.416873,NaN,NaN,NaN,NaN
71,LinearSVM,{'C': 0.1},"LinearSVM|{""C"": 0.1}",0.10,NaN,0.822906,0.855215,0.887394,0.871007,0.154570,NaN,NaN,NaN,NaN
80,LinearSVM,{'C': 1.25},"LinearSVM|{""C"": 1.25}",1.25,NaN,0.824872,0.866480,0.874897,0.870668,0.315552,NaN,NaN,NaN,NaN
79,LinearSVM,{'C': 1.2},"LinearSVM|{""C"": 1.2}",1.20,NaN,0.824842,0.866342,0.875034,0.870666,0.325236,NaN,NaN,NaN,NaN



----- LinearSVM Loss Check tuning started -----
Total configs for LinearSVM Loss Check: 28
Skipping LinearSVM Loss Check 1/28: already saved
Skipping LinearSVM Loss Check 2/28: already saved
Skipping LinearSVM Loss Check 3/28: already saved
Skipping LinearSVM Loss Check 4/28: already saved
Skipping LinearSVM Loss Check 5/28: already saved
Skipping LinearSVM Loss Check 6/28: already saved
Skipping LinearSVM Loss Check 7/28: already saved
Skipping LinearSVM Loss Check 8/28: already saved
Skipping LinearSVM Loss Check 9/28: already saved
Skipping LinearSVM Loss Check 10/28: already saved
Skipping LinearSVM Loss Check 11/28: already saved
Skipping LinearSVM Loss Check 12/28: already saved
Skipping LinearSVM Loss Check 13/28: already saved
Skipping LinearSVM Loss Check 14/28: already saved
Skipping LinearSVM Loss Check 15/28: already saved
Skipping LinearSVM Loss Check 16/28: already saved
Skipping LinearSVM Loss Check 17/28: already saved
Skipping LinearSVM Loss Check 18/28: already saved

,model,params,key,C,l1_ratio,accuracy,precision,recall,f1,time_min,alpha,fit_prior,norm,loss
94,LinearSVM Loss Check,"{'C': 0.75, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 0.75, ""loss"": ""hinge""}",0.75,NaN,0.829267,0.865206,0.884384,0.874690,0.235294,NaN,NaN,NaN,hinge
96,LinearSVM Loss Check,"{'C': 0.8, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 0.8, ""loss"": ""hinge""}",0.80,NaN,0.829021,0.865223,0.883928,0.874475,0.195913,NaN,NaN,NaN,hinge
110,LinearSVM Loss Check,"{'C': 1.75, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 1.75, ""loss"": ""hinge""}",1.75,NaN,0.829205,0.867226,0.881465,0.874288,0.389258,NaN,NaN,NaN,hinge
108,LinearSVM Loss Check,"{'C': 1.5, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 1.5, ""loss"": ""hinge""}",1.50,NaN,0.829052,0.866768,0.881830,0.874234,0.182446,NaN,NaN,NaN,hinge
98,LinearSVM Loss Check,"{'C': 0.9, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 0.9, ""loss"": ""hinge""}",0.90,NaN,0.828529,0.865584,0.882560,0.873989,0.154115,NaN,NaN,NaN,hinge
102,LinearSVM Loss Check,"{'C': 1.1, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 1.1, ""loss"": ""hinge""}",1.10,NaN,0.828591,0.866153,0.881875,0.873944,0.304934,NaN,NaN,NaN,hinge
100,LinearSVM Loss Check,"{'C': 1, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 1, ""loss"": ""hinge""}",1.00,NaN,0.828529,0.865912,0.882103,0.873932,0.142331,NaN,NaN,NaN,hinge
104,LinearSVM Loss Check,"{'C': 1.2, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 1.2, ""loss"": ""hinge""}",1.20,NaN,0.828499,0.866332,0.881465,0.873833,0.196939,NaN,NaN,NaN,hinge
106,LinearSVM Loss Check,"{'C': 1.25, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 1.25, ""loss"": ""hinge""}",1.25,NaN,0.828376,0.866111,0.881556,0.873765,0.154011,NaN,NaN,NaN,hinge
112,LinearSVM Loss Check,"{'C': 2, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 2, ""loss"": ""hinge""}",2.00,NaN,0.828468,0.867149,0.880279,0.873665,0.248967,NaN,NaN,NaN,hinge


### Training

Fit the Linear Support Vector Machine model using the engineered combined feature representation of the training data.

In [17]:
# reused seed used in data split script
svm = LinearSVC(max_iter=5000, C=1.5, random_state=42, loss='hinge')
svm.fit(train_features, train_labels)

,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'hinge'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.5
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forthe dual coordinate descent (if ``dual=True``). When ``dual=False`` theunderlying implementation of :class:`LinearSVC` is not random and``random_state`` has no effect on the results.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_iter max_iter: int, default=1000The maximum number of iterations to be run.",5000
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed t

### Threshold Selection

The default Linear SVM threshold is 0.0 because its decision scores are margins rather than probabilities.

A negative selected threshold is valid, it shifts the operating point so that examples with slightly negative margins are also classified as PII. This generally increases PII recall and reduces false negatives while increasing false positives.

In [18]:
svm_scores = svm.decision_function(val_features)
svm_threshold = find_best_threshold(val_labels, svm_scores)[0]
print('Best threshold: ', svm_threshold)
display(find_best_threshold(val_labels, svm_scores)[1].head(10))

Best threshold:  -0.1773982139911725


,threshold,precision,recall,f1
9067,-0.177398,0.847746,0.907598,0.876652
9101,-0.172358,0.848336,0.906914,0.876648
9078,-0.176388,0.847931,0.907370,0.876644
9071,-0.176912,0.847806,0.907507,0.876641
9080,-0.176025,0.847960,0.907325,0.876639
9098,-0.172633,0.848270,0.906960,0.876634
9066,-0.177624,0.847710,0.907598,0.876633
9100,-0.172429,0.848300,0.906914,0.876628
8849,-0.212114,0.843956,0.911931,0.876628
9068,-0.177263,0.847740,0.907553,0.876627


### Evaluation

Generate predictions on the validation set for model evaluation, then evaluate model performance using a classification report and confusion matrix.

In [19]:
svm_predictions = (svm_scores >= svm_threshold).astype(int)
print(classification_report(val_labels, svm_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(val_labels, svm_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.78      0.66      0.72     10616
         PII       0.85      0.91      0.88     21926

    accuracy                           0.83     32542
   macro avg       0.81      0.79      0.80     32542
weighted avg       0.82      0.83      0.82     32542



,Predicted Safe,Predicted PII
Actual Safe,7042,3574
Actual PII,2026,19900


# Final Held Out Test Evaluation

After the model hyperparameters and decision thresholds have been selected ont eh validation split, every configuration is frozen and evaluated on the held out test split.

#### Majority Class Baseline

The always PII baseline is repeated on the test split to provide the same reference point used during validation.

In [20]:
baseline_test_predictions = np.ones(len(test_labels), dtype=int)
# zero_division set to 0 to hide warning due to zero examples being predicted as safe
print(classification_report(test_labels, baseline_test_predictions, zero_division=0))

cm = confusion_matrix(test_labels, baseline_test_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     10617
           1       0.67      1.00      0.81     21926

    accuracy                           0.67     32543
   macro avg       0.34      0.50      0.40     32543
weighted avg       0.45      0.67      0.54     32543



,Predicted Safe,Predicted PII
Actual Safe,0,10617
Actual PII,0,21926


#### Logistic Regression

In [21]:
lr_scores = lr.predict_proba(test_features)[:, 1]
lr_test_predictions = (lr_scores >= lr_threshold).astype(int)
print(classification_report(test_labels, lr_test_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(test_labels, lr_test_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.79      0.66      0.72     10617
         PII       0.85      0.92      0.88     21926

    accuracy                           0.83     32543
   macro avg       0.82      0.79      0.80     32543
weighted avg       0.83      0.83      0.83     32543



,Predicted Safe,Predicted PII
Actual Safe,7012,3605
Actual PII,1841,20085


#### Naive Bayes

In [22]:
nb_scores = nb.predict_proba(test_features)[:, 1]
nb_test_predictions = (nb_scores >= nb_threshold).astype(int)
print(classification_report(test_labels, nb_test_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(test_labels, nb_test_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.69      0.34      0.46     10617
         PII       0.74      0.92      0.82     21926

    accuracy                           0.74     32543
   macro avg       0.72      0.63      0.64     32543
weighted avg       0.73      0.74      0.71     32543



,Predicted Safe,Predicted PII
Actual Safe,3658,6959
Actual PII,1650,20276


#### Linear SVM

In [23]:
svm_scores = svm.decision_function(test_features)
svm_test_predictions = (svm_scores >= svm_threshold).astype(int)
print(classification_report(test_labels, svm_test_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(test_labels, svm_test_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.79      0.67      0.72     10617
         PII       0.85      0.91      0.88     21926

    accuracy                           0.83     32543
   macro avg       0.82      0.79      0.80     32543
weighted avg       0.83      0.83      0.83     32543



,Predicted Safe,Predicted PII
Actual Safe,7110,3507
Actual PII,1931,19995


## Save Evaluation

In [24]:
# load the original test prompts
test_df = pd.read_parquet(repo_root / 'data_splits/test.parquet').reset_index(drop=True)

classical_predictions = {'Logistic Regression': lr_test_predictions, 'Naive Bayes': nb_test_predictions, 'Linear SVM': svm_test_predictions}

# save confusion matrix rows
confusion_rows = []

for model_name, model_predictions in classical_predictions.items():
    tn, fp, fn, tp = confusion_matrix(test_labels, model_predictions).ravel()
    confusion_rows.append({
        'model': model_name,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp
    })

pd.DataFrame(confusion_rows).to_csv(repo_root / 'results/classical_confusion_matrices.csv', index=False)


# save test predictions
# scores included so 06 can re-threshold / rank errors, not just count them
test_prediction_results = pd.DataFrame({
    'original_index': test_df['original_index'].to_numpy(),
    'true_label': test_labels,
    'logistic_regression': lr_test_predictions,
    'naive_bayes': nb_test_predictions,
    'linear_svm': svm_test_predictions,
    'logistic_regression_score': lr_scores,
    'naive_bayes_score': nb_scores,
    'linear_svm_score': svm_scores
})

test_prediction_results.to_csv(repo_root / 'results/classical_test_predictions.csv', index=False)

## Error Analysis

The confusion matrices show how many errors each model makes, but reviewing the individual examples helps give insight into the types of prompts that are misclassified.

The following cell reports the false negative and false positive counts for each classical model and prints examples of each error type.

In [25]:
# confirm the prompts and labels are aligned
assert len(test_df) == len(test_labels)
assert np.array_equal(test_df['label'].to_numpy(), test_labels)

model_predictions = {'Logistic Regression': lr_test_predictions, 'Naive Bayes': nb_test_predictions, 'Linear SVM': svm_test_predictions}
errors = []

for model_name, predictions in model_predictions.items():
    model_errors = test_df[['text', 'label']].copy()
    model_errors['prediction'] = predictions

    fns = model_errors[(model_errors['label'] == 1) & (model_errors['prediction'] == 0)]
    fps = model_errors[(model_errors['label'] == 0) & (model_errors['prediction'] == 1)]

    errors.append({
        'Model': model_name,
        'False Negatives': len(fns),
        'False Positives': len(fps),
        'Median FN length': fns['text'].str.len().median(),
        'Median FP length': fps['text'].str.len().median()
    })

    print(f'\n------------------ {model_name} ------------------')
    print(f'False negatives: {len(fns)}')
    print(f'False positives: {len(fps)}')

    print('\nFalse negative examples:')
    for text in fns['text'].sample(10, random_state=42):
        print(repr(text[:200]))

    print('\nFalse positive examples:')
    for text in fps['text'].sample(10, random_state=42):
        print(repr(text[:200]))

pd.DataFrame(errors)


------------------ Logistic Regression ------------------
False negatives: 1841
False positives: 3605

False negative examples:
'De volledige transfer van rechten betreffende het werk van Pierluigi is vastgelegd onder PFNN907844. Het registratieadres is 30895xC8fDpdmy8faqBIsqA7nS8VhyUaVp.'
'Les données historiques montrent une variation de primes pour les utilisateurs ayant un solde de 787323 Franc suisse. Les ajustements se basent sur des profils comme 0x3145e93600ebb684b062c748469bbeab'
'<p>Attention to Nodar</p><p>Study: Contract</p><p>Date/Time: 21</p><p>Bank: Aldermore Bank</p>'
'Sono Conto Corrente da 59100. Leggi le mie riflessioni sul forum di supporto peer-to-peer: http://vespucci.net/.'
"CA residents, anyone suggest a therapist? I've lost balance again after my 14/12/2019. What’s your experience with Revolut covering the cost?"
'Les infections détectées dans le quartier de Ledistrasse sont détaillées dans le document REF-201108-7388. Agissez rapidement pour prévenir la propa

,Model,False Negatives,False Positives,Median FN length,Median FP length
0,Logistic Regression,1841,3605,124.0,132.0
1,Naive Bayes,1650,6959,133.0,119.0
2,Linear SVM,1931,3507,124.0,132.0


### Error Analysis Findings

Many false negatives appeared in multilingual prompts and contained structured or synthetic identifiers embedded within longer sentences. These included financial identifiers, cryptocurrency addresses, coordinates, document references, IP placeholders, usernames, and other alphanumeric values. Some were also embedded in HTML formatted text. This suggests that varied identifier formats and multilingual context remain challenging for the classical models.

Logistic Regression and Linear SVM showed similar qualitative error patterns and comparable error counts. Both missed prompts containing structured values or sensitive information embedded in multilingual text. Naive Bayes produced fewer false negatives, but substantially more false positives. This indicates a tendency to classify ambiguous prompts as privacy sensitive.

The false positive samples showed that the models sometimes confused PII related language with actual PII. Several safe prompts requested sensitive information, discussed PII-related concepts, or contained structured values that were not labeled as PII. Since the task classifies prompts based on whether PII is actually present, requesting an identifier does not make a prompt privacy sensitive when no sensitive value is disclosed.

## Per Prompt Inference Latency

To evaluate responsiveness, the classical models were benchmarked on the same random sample of 100 prompts from the test set.

The reported latency is end to end, including the TF-IDF transformation, text/pattern feature extraction, combining the features, model prediction, and thresholding.

In [26]:
# load the fitted preprocessing objects; the .joblib artifacts are local to 02 (not
# committed), so refit from the frozen train split on a clean clone (same config, train-only)
_joblibs = [FEATURE_DIR / 'tfidf_vectorizer.joblib',
            FEATURE_DIR / 'text_feature_scaler.joblib',
            FEATURE_DIR / 'pattern_feature_scaler.joblib']
if all(p.exists() for p in _joblibs):
    tfidf_vectorizer = joblib.load(_joblibs[0])
    text_feature_scaler = joblib.load(_joblibs[1])
    pattern_feature_scaler = joblib.load(_joblibs[2])
else:
    print('transformer .joblib files not found -> refitting from the frozen train split ...')
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import MinMaxScaler
    _tr = pd.read_parquet(repo_root / 'data_splits/train.parquet')['text'].fillna('')
    tfidf_vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_features=50000).fit(_tr)
    text_feature_scaler = MinMaxScaler().fit(extract_text_features(_tr))
    pattern_feature_scaler = MinMaxScaler().fit(extract_pattern_features(_tr))

# converts a raw prompt into the combined feature representation
def featurize_prompt(text):
    text_series = pd.Series([text])

    tfidf_features = tfidf_vectorizer.transform(text_series)

    text_features = extract_text_features(text_series)
    text_features = text_feature_scaler.transform(text_features)
    text_features = csr_matrix(text_features)

    pattern_features = extract_pattern_features(text_series)
    pattern_features = pattern_feature_scaler.transform(pattern_features)
    pattern_features = csr_matrix(pattern_features)


    return hstack([tfidf_features, text_features, pattern_features], format='csr')

# select 100 samples
timing_indices = test_df.sample(
    n=100,
    random_state=42
).index.to_numpy()

# warmup to ensure initialization doesn't mess up timing
warmup_text = test_df.loc[timing_indices[0], 'text']
warmup_features = featurize_prompt(warmup_text)

lr.predict_proba(warmup_features)
nb.predict_proba(warmup_features)
svm.decision_function(warmup_features)

latency = {}


# Logistic Regression
lr_latencies = []

for index in timing_indices:
    text = test_df.loc[index, 'text']

    start = time.perf_counter()

    feature_row = featurize_prompt(text)
    score = lr.predict_proba(feature_row)[0, 1]
    prediction = int(score >= lr_threshold)

    lr_latencies.append((time.perf_counter() - start) * 1000)

latency['Logistic Regression'] = {
    'mean_ms': np.mean(lr_latencies),
    'median_ms': np.median(lr_latencies),
    'min_ms': np.min(lr_latencies),
    'max_ms': np.max(lr_latencies),
}


# Naive Bayes
nb_latencies = []

for index in timing_indices:
    text = test_df.loc[index, 'text']

    start = time.perf_counter()

    feature_row = featurize_prompt(text)
    score = nb.predict_proba(feature_row)[0, 1]
    prediction = int(score >= nb_threshold)

    nb_latencies.append((time.perf_counter() - start) * 1000)

latency['Naive Bayes'] = {
    'mean_ms': np.mean(nb_latencies),
    'median_ms': np.median(nb_latencies),
    'min_ms': np.min(nb_latencies),
    'max_ms': np.max(nb_latencies),
}


# Linear SVM
svm_latencies = []

for index in timing_indices:
    text = test_df.loc[index, 'text']

    start = time.perf_counter()

    feature_row = featurize_prompt(text)
    score = svm.decision_function(feature_row)[0]
    prediction = int(score >= svm_threshold)

    svm_latencies.append((time.perf_counter() - start) * 1000)

latency['Linear SVM'] = {
    'mean_ms': np.mean(svm_latencies),
    'median_ms': np.median(svm_latencies),
    'min_ms': np.min(svm_latencies),
    'max_ms': np.max(svm_latencies),
}
latency_df = pd.DataFrame(latency).T

# match current latency_benchmark.csv stuff
latency_df.index.name = 'model'
latency_df.insert(0, 'n', len(timing_indices))
latency_df = latency_df.reset_index()

latency_path = repo_root / 'results/latency_benchmark.csv'
latency_benchmark = pd.read_csv(latency_path)

# remove old classical rows before adding the new measurements
latency_benchmark = latency_benchmark[~latency_benchmark['model'].isin(latency_df['model'])]

latency_benchmark = pd.concat([latency_benchmark, latency_df], ignore_index=True)

latency_benchmark.to_csv(latency_path, index=False)

latency_df.round(4)

,model,n,mean_ms,median_ms,min_ms,max_ms
0,Logistic Regression,100,1.5486,1.5423,1.4885,1.7178
1,Naive Bayes,100,1.6852,1.6826,1.6245,1.7659
2,Linear SVM,100,1.5124,1.5113,1.4659,1.5830


## Test Set Results Summary Chart

In [27]:
predictions = {
    'Baseline': baseline_test_predictions,
    'Logistic Regression': lr_test_predictions,
    'Naive Bayes': nb_test_predictions,
    'Linear SVM': svm_test_predictions
}

comparison_results = []

for model, p in predictions.items():
    model_latency = latency.get(model, {})
    comparison_results.append({
        'Model': model,
        'Accuracy': accuracy_score(test_labels, p),
        'PII Precision': precision_score(test_labels, p, zero_division=0),
        'PII Recall': recall_score(test_labels, p, zero_division=0),
        'PII F1': f1_score(test_labels, p, zero_division=0),
        'Macro F1': f1_score(test_labels, p, average='macro', zero_division=0),
        "Average ms/prompt": model_latency.get("mean_ms", np.nan),
        "Median ms/prompt": model_latency.get("median_ms", np.nan),
        "Minimum ms/prompt": model_latency.get("min_ms", np.nan),
        "Maximum ms/prompt": model_latency.get("max_ms", np.nan)
    })

comparison_results = pd.DataFrame(comparison_results).round(4)

comparison_results

,Model,Accuracy,PII Precision,PII Recall,PII F1,Macro F1,Average ms/prompt,Median ms/prompt,Minimum ms/prompt,Maximum ms/prompt
0,Baseline,0.6738,0.6738,1.0000,0.8051,0.4025,NaN,NaN,NaN,NaN
1,Logistic Regression,0.8327,0.8478,0.9160,0.8806,0.8004,1.5486,1.5423,1.4885,1.7178
2,Naive Bayes,0.7355,0.7445,0.9247,0.8249,0.6421,1.6852,1.6826,1.6245,1.7659
3,Linear SVM,0.8329,0.8508,0.9119,0.8803,0.8018,1.5124,1.5113,1.4659,1.5830


## Final Results and Key Findings

The final classical models were evaluated on the held out test set of 32,543 prompts.

**Selected model:** Logistic Regression

**Configuration:** `C=2.65, l1_ratio=1, random_state=42, max_iter=1000, solver='liblinear'`

**Threshold:** 0.41589660496670616

It achieved **0.8327** accuracy, **0.8478** PII precision, **0.9160** PII recall, **0.8806** PII F1, and **0.8004** macro F1, with **3605** false positives and **1841** false negatives.

### Key Findings

- Logistic Regression achieved the strongest overall classical performance, providing the best balance between PII precision and recall.
- Linear SVM performed nearly identically, with a PII F1 of **0.8803**, suggesting that both linear classifiers extracted similar predictive information from the shared feature representation.
- Naive Bayes achieved the highest PII recall, but its lower precision produced substantially more false positives. This indicated a stronger tendency to classify safe prompts as containing PII.
- Validation selected threshold tuning shifted the models toward detecting more PII prompts, reducing false negatives at the cost of additional false positives and lower safe class recall.

### Limitations and Error Patterns

- **False positive tendency**: The classical models generally favored PII recall over safe class performance, causing some additional safe prompts to be incorrectly flagged as containing PII. THis pattern was most pronounced for Multinomial Naive Bayes which achieved both the highest PII recall and false positives.
- **Contextual modeling**: TF-IDF and manually engineered features capture the character and structural patterns but do not directly serve to model the contextual meaning. The models may struggle with contextual, multilingual, or obfuscated PII.
- **Unified feature representation**: All classical models used the same feature matrix to support a controlled comparison. This representation may not be individually optimal for every model.
- **Single validation split:** Hyperparameters and thresholds were selected using the fixed validation split rather than cross validation. This maintained computational efficiency and consistency across the team pipelines, but the selected settings may be somewhat dependent on the specific split.


## Feature-Set Ablation (project plan §8: define **and compare** Feature Sets 1–4)

The combined matrix is `hstack([tfidf | text | pattern])`, so each block is an exact **column slice**
of the already-loaded matrices — no rebuild needed. The **Combined row reuses the canonical `lr` model and threshold** (asserted equal to the
main-table export), so the two tables cannot disagree; the three block rows `clone()` the same
constructor, re-tune the decision threshold on validation (thresholds do not transfer
across feature spaces), and evaluate once on the held-out test set. The result quantifies the marginal
value of each engineered block; saved to `results/feature_ablation.csv` for the write-up (00 §9.1).


In [28]:
# feature-set ablation: tuned LR on each feature block (exact column slices of combined).
# CONSISTENCY RULES (review finding): the Combined row REUSES the canonical `lr` model and
# `lr_threshold` — it is the main-table row by construction, not a refit. The three block
# rows clone cell 12's constructor VERBATIM (no added penalty argument — an explicit
# penalty='l2' previously produced a slightly different model than the canonical fit) and
# use the notebook's own find_best_threshold().
from sklearn.base import clone

N_META = 18                                        # 10 text-shape + 8 pattern columns
tfidf_width = train_features.shape[1] - N_META
BLOCKS = {
    "TF-IDF only":        slice(0, tfidf_width),
    "Text features only": slice(tfidf_width, tfidf_width + 10),
    "Patterns only":      slice(tfidf_width + 10, tfidf_width + 18),
}

ablation_rows = []
for block_name, cols in BLOCKS.items():
    Xtr, Xva, Xte = train_features[:, cols], val_features[:, cols], test_features[:, cols]
    ab_lr = clone(lr).fit(Xtr, train_labels)               # identical config to cell 12
    thr = find_best_threshold(val_labels, ab_lr.predict_proba(Xva)[:, 1])[0]
    te_pred = (ab_lr.predict_proba(Xte)[:, 1] >= thr).astype(int)
    ablation_rows.append({
        "feature_set": block_name, "n_features": Xtr.shape[1], "val_threshold": round(float(thr), 4),
        "test_pii_precision": round(precision_score(test_labels, te_pred, zero_division=0), 4),
        "test_pii_recall": round(recall_score(test_labels, te_pred), 4),
        "test_pii_f1": round(f1_score(test_labels, te_pred), 4),
        "test_macro_f1": round(f1_score(test_labels, te_pred, average="macro"), 4),
        "test_fn": int(((test_labels == 1) & (te_pred == 0)).sum()),
        "test_fp": int(((test_labels == 0) & (te_pred == 1)).sum()),
    })
    print(f"{block_name:20s} done ({Xtr.shape[1]:,} features, threshold {thr:.3f})")

# Combined row = the canonical model at the canonical threshold (no refit)
comb_pred = (lr.predict_proba(test_features)[:, 1] >= lr_threshold).astype(int)
comb_row = {
    "feature_set": "Combined (canonical LR)", "n_features": train_features.shape[1],
    "val_threshold": round(float(lr_threshold), 4),
    "test_pii_precision": round(precision_score(test_labels, comb_pred, zero_division=0), 4),
    "test_pii_recall": round(recall_score(test_labels, comb_pred), 4),
    "test_pii_f1": round(f1_score(test_labels, comb_pred), 4),
    "test_macro_f1": round(f1_score(test_labels, comb_pred, average="macro"), 4),
    "test_fn": int(((test_labels == 1) & (comb_pred == 0)).sum()),
    "test_fp": int(((test_labels == 0) & (comb_pred == 1)).sum()),
}
ablation_rows.append(comb_row)

# guard: the Combined row must equal the exported main-table confusion matrix exactly
_cm = pd.read_csv(repo_root / "results" / "classical_confusion_matrices.csv").set_index("model")
assert comb_row["test_fn"] == int(_cm.loc["Logistic Regression", "fn"]), \
    "ablation Combined row disagrees with classical_confusion_matrices.csv — re-export both from one run"
assert comb_row["test_fp"] == int(_cm.loc["Logistic Regression", "fp"]), \
    "ablation Combined row disagrees with classical_confusion_matrices.csv — re-export both from one run"
print("Combined row verified identical to the main-table export.")

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(repo_root / "results" / "feature_ablation.csv", index=False)
print("saved results/feature_ablation.csv")
ablation_df


TF-IDF only          done (50,000 features, threshold 0.429)
Text features only   done (10 features, threshold 0.461)
Patterns only        done (8 features, threshold 0.506)
Combined row verified identical to the main-table export.
saved results/feature_ablation.csv


,feature_set,n_features,val_threshold,test_pii_precision,test_pii_recall,test_pii_f1,test_macro_f1,test_fn,test_fp
0,TF-IDF only,50000,0.4293,0.8329,0.9058,0.8678,0.7773,2066,3984
1,Text features only,10,0.4609,0.6944,0.9683,0.8088,0.5054,694,9346
2,Patterns only,8,0.5063,0.6738,1.0000,0.8051,0.4026,0,10616
3,Combined (canonical LR),50018,0.4159,0.8478,0.9160,0.8806,0.8004,1841,3605
